In [ ]:
import sys
import os
import uproot
import pandas as pd
import numpy as np
import awkward as ak
import math

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
matplotlib.use('QtAgg')
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qtagg import NavigationToolbar2QT as NavigationToolbar
from matplotlib.figure import Figure
from matplotlib.animation import FuncAnimation
from mpl_toolkits.axes_grid1 import make_axes_locatable

from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout, 
    QComboBox, QLabel, QSpinBox, QPushButton, QStackedWidget, QGroupBox,
    QTableWidget, QTableWidgetItem, QHeaderView, QCheckBox, QButtonGroup,
    QRadioButton, QLineEdit
)
from PyQt6.QtCore import Qt, QTimer

CMTOMICRON = 1e4
VCMTOkVCM = 1e-3

In [ ]:
class AnimationData:
    DEFAULT_FILENAME = 'animationData.root'

    def __init__(self, fileName: str = None):
        self.fileName = fileName if fileName else self.DEFAULT_FILENAME
        self.simData = None
        self.avalancheData = None
        self.fieldStengths = None
        self.fieldLines = None
        self.particleData = None
        self.signalData = None
        
        self.loadRootData()

        return

#**********************************************************************#
    def loadRootData(self):
        """
        Reads ROOT trees into Pandas DataFrames.
        """
        dataPath = './Data/'
        filePath = os.path.join(dataPath, self.fileName)
        
        if not os.path.exists(filePath):
            print(f"Warning: '{filePath}' does not exist.")
            return

        with uproot.open(filePath) as file:

            # Simulation Metadata
            if 'simDataTree' in file:
                simDataDF = {
                    k: v[0]
                    for k, v in file['simDataTree'].arrays(library='np').items()
                }
                geoKeys = [
                    'padLength', 'pitch', 'holeRadius',
                    'amplificationGap', 'driftLength',
                    'gridThickness', 'padThickness',
                    'thicknessSiO2', 'pillarRadius'
                ]
                for key in geoKeys:
                    if key in simDataDF:
                        simDataDF[key] *= CMTOMICRON
                if 'driftField' in simDataDF:
                    simDataDF['driftField'] *= VCMTOkVCM
                self.simData = simDataDF

            # Avalanche Overview Data
            if 'avalancheTree' in file:
                self.avalancheData = file['avalancheTree'].arrays(
                    ['AvalancheID', 'Gain'], library='pd'
                )

            # Electric & Weighting Fields
            if 'fieldTree' in file:
                fieldDF = file['fieldTree'].arrays(library='pd')
                fieldDF[['x', 'y', 'z']] *= CMTOMICRON
                fieldDF[['Ex', 'Ey', 'Ez']] *= VCMTOkVCM
                fieldDF['E'] = np.linalg.norm(
                    fieldDF[['Ex', 'Ey', 'Ez']].values, axis=1
                )
                self.fieldStrengths = fieldDF

            # Electric Field Lines
            if 'fieldLineTree' in file:
                lineDF = file['fieldLineTree'].arrays(
                    ['FieldLineID', 'FieldStart', 'x', 'y', 'z'], library='pd'
                )
                lineDF[['x', 'y', 'z']] *= CMTOMICRON
                self.fieldLines = lineDF

            # Particle Tracks (Awkward Array -> Flattened DataFrame)
            if 'particleDataTree' in file:
                pData = file['particleDataTree'].arrays(
                    [
                        'AvalancheID', 'FrameID',
                        'Time', 'ParticleType',
                        'x', 'y', 'z',
                    ],
                    library='ak',
                )
                particleDF = self._flattenBranch(pData)
                particleDF[['x', 'y', 'z']] *= CMTOMICRON
                self.particleData = particleDF

            # Induced Signal Traces
            if 'signalDataTree' in file:
                self.signalData = file['signalDataTree'].arrays(library='pd')

        return
    
#**********************************************************************#
    @staticmethod
    def _flattenBranch(pData) -> pd.DataFrame:
        """
        Flattens nested C++ std::vector particle branches.
        """

        flatAvalanche = ak.flatten(ak.broadcast_arrays(pData['AvalancheID'], pData['x'])[0])
        flatFrame = ak.flatten(ak.broadcast_arrays(pData['FrameID'], pData['x'])[0])
        flatTime = ak.flatten(ak.broadcast_arrays(pData['Time'], pData['x'])[0])

        allData = {
            'AvalancheID': ak.to_numpy(flatAvalanche),
            'FrameID': ak.to_numpy(flatFrame),
            'Time': ak.to_numpy(flatTime),
            'ParticleType': ak.to_numpy(ak.flatten(pData['ParticleType'])),
            'x': ak.to_numpy(ak.flatten(pData['x'])),
            'y': ak.to_numpy(ak.flatten(pData['y'])),
            'z': ak.to_numpy(ak.flatten(pData['z']))
        }
        return pd.DataFrame(allData)

In [ ]:
allData = AnimationData()